In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import geopandas as gpd
import urllib
import sys
import pathlib
import glob
import matplotlib.pyplot as plt
from urllib.parse import quote
from sqlalchemy import create_engine
import configparser
import requests

import statsmodels.api as sm
#import pingouin as pg

import plotly.express as px
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


from fluxdataqaqc import Data, QaQc, Plot
from bokeh.plotting import figure, show, ColumnDataSource
from bokeh.models.formatters import DatetimeTickFormatter
from bokeh.models import LinearAxis, Range1d
from bokeh.io import output_notebook
output_notebook()


In [ ]:
sys.path.append("../../src/")
from micromet import AmerifluxDataProcessor
#from tests.ffp_tests import footprint_config

In [ ]:
config = configparser.ConfigParser()

config.read('../../secrets/config.ini')

from sqlalchemy import create_engine
import urllib.parse
host = config['DEFAULT']['ip']
pw = config['DEFAULT']['pw']
user = config['DEFAULT']['login']

encoded_password = urllib.parse.quote_plus(pw)

def postconn_et(encoded_password, host='localhost',user='postgres',port='5432',db='groundwater', schema = 'groundwater'):
    connection_text = "postgresql+psycopg2://{:}:{:}@{:}:{:}/{:}?gssencmode=disable".format(user,encoded_password,host,port,db)
    return create_engine(connection_text, connect_args={'options': '-csearch_path={}'.format(schema)})


engine = postconn_et(encoded_password, host=host, user=user)

In [ ]:
sql = "SELECT * from groundwater.micromet_merged_view where stationid='US-UTD'"
df = pd.read_sql(sql, engine, )#index_col='datetime_start', parse_dates=True).replace(-9999, np.nan)
df

In [ ]:
sql = "SELECT * from groundwater.micromet_merged_view"
df = pd.read_sql(sql, engine, )#index_col='datetime_start', parse_dates=True).replace(-9999, np.nan)
df.to_parquet("G:/Shared drives/UGS_Flux/Data_Downloads/compiled/old_database_merged_view.parquet")

In [ ]:
sql = "SELECT * from groundwater.micromet_eddy"
df = pd.read_sql(sql, engine, )#index_col='datetime_start', parse_dates=True).replace(-9999, np.nan)
df.to_parquet("G:/Shared drives/UGS_Flux/Data_Downloads/compiled/old_database_eddy.parquet")

In [ ]:
sql = "SELECT * from groundwater.micromet_met"
df = pd.read_sql(sql, engine, )#index_col='datetime_start', parse_dates=True).replace(-9999, np.nan)
df.to_parquet("G:/Shared drives/UGS_Flux/Data_Downloads/compiled/old_database_met.parquet")

In [ ]:
for col in df.columns:
    print(col)

In [ ]:
sql = "SELECT * from groundwater.micromet_eddy where stationid='US-UTM'"
df = pd.read_sql(sql, engine, index_col='datetime_start', parse_dates=True).replace(-9999, np.nan)

df

In [ ]:
import plotly.graph_objects as go

# ----- assume `hourly` already exists from the resample/agg step -----
# If not, run the block from the previous answer first.

fig = go.Figure()

# 1) add the “central-tendency” curves
fig.add_scatter(
    x=hourly.index, y=hourly["mean"],
    mode="lines", name="mean"
)

In [ ]:
config_path = '../../station_config/US-UTP.ini'
d = Data(config_path)
q = QaQc(d)

In [ ]:
q.monthly_df

In [ ]:
d.df
d.inv_map

In [ ]:
# convert to internal names, copy dataframe
df = d.df.rename(columns=d.inv_map)
# day of year mean of input energy balance components
vars_we_want = ['H', 'LE', 'Rn']
doy_means = df[vars_we_want].groupby(d.df.index.dayofyear).mean()
# create a Bokeh figure
fig = figure(x_axis_label='day of year', y_axis_label='day of year mean (w/m2)')
# arguements needed for creating interactive plots
plt_vars = vars_we_want
colors = ['red', 'blue', 'black', 'green']
x_name = 'date'
source = ColumnDataSource(doy_means)
Plot.add_lines(fig, doy_means, plt_vars, colors, x_name, source, labels=vars_we_want,
    x_axis_type=None)
show(fig)

In [ ]:
station = 'US-UTP'
startdate = '2020-01-01'

headdict = {'Accept-Profile': 'groundwater','Content-Type': 'application/json'}

params = {'stationid':f'eq.{station}',
         'datetime_start':f'gte.{startdate}'
         }

resp_ed = requests.get("https://ugs-koop-umfdxaxiyq-wm.a.run.app/amfluxeddy",headers=headdict,params=params)
resp_met = requests.get("https://ugs-koop-umfdxaxiyq-wm.a.run.app/amfluxmet",headers=headdict,params=params)

In [ ]:
resp_met

In [ ]:
data_ed = pd.DataFrame(resp_ed.json())
data_met = pd.DataFrame(resp_met.json())

if len(data_ed) > 0:
    data_ed['datetime_start'] = pd.to_datetime(data_ed['datetime_start'])
    data_ed = data_ed.set_index(['stationid','datetime_start'])

if len(data_met) > 0:
    data_met['datetime_start'] = pd.to_datetime(data_met['datetime_start'])
    data_met = data_met.set_index(['stationid','datetime_start'])

if len(data_ed) > 0 and len(data_met) > 0:
    all_data = pd.concat([data_ed,data_met],axis=1)

In [ ]:
data_ed.replace(-9999,np.nan,inplace=True)

In [ ]:
sql = "SELECT * FROM groundwater.amfluxeddy WHERE stationid = 'US-UTP' and datetime_start >= '2024-06-07 9:00'"
data = pd.read_sql(sql,engine)
data.replace(-9999,np.nan,inplace=True)
data.set_index('datetime_start',inplace=True)
#data = data.loc[pd.to_datetime('2024-06-07 9:00'):]

# Replace values in column 'A' below 0 with NaN
data['et'] = np.where(data['et'] < 0, np.nan, data['et'])
hrly = data.resample('1h').mean(numeric_only=True)
hrly = hrly.interpolate(method='time',limit=4)
# Compute daily cumulative sum
hrly['et_cumsum'] = hrly.groupby(hrly.index.date)['et'].cumsum()

hrly['et'].plot()

In [ ]:
# Compute daily cumulative sum
dlyet = hrly.groupby(pd.Grouper(level=0,freq='D'))['et'].sum().to_frame()
dlyet['month'] = dlyet.index.month
import seaborn as sns
# Plot boxplots by month
plt.figure(figsize=(12, 6))
sns.boxplot(x='month', y='et', data=dlyet)
plt.title('Boxplots of Daily Values by Month')
plt.xlabel('Month')
plt.ylabel('Daily Values')
#plt.xticks(ticks=range(12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.show()


In [ ]:
6*30*0.0393701

In [ ]:
stat_config = configparser.ConfigParser()
stat_config.read('../../station_config/US-UTP.ini')

latitude = stat_config['METADATA']['station_latitude']
longitude = stat_config['METADATA']['station_longitude']

crs = 4326

import pygridmet as gridmet
coords = (longitude, latitude)
dates = ("2024-06-07", "2024-12-27")

var = ['sph', 'srad', 'etr', 'pet', 'vs', 'tmmx', 'tmmn', 'vpd']

gridmet_data = gridmet.get_bycoords(coords, dates, variables=var, crs=crs)

<table class="table">
<thead>
<tr class="row-odd"><th class="head"><p>Variable</p></th>
<th class="head"><p>Abbr</p></th>
<th class="head"><p>Unit</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p>Precipitation</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">pr</span></code></p></td>
<td><p>mm</p></td>
</tr>
<tr class="row-odd"><td><p>Maximum Relative Humidity</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">rmax</span></code></p></td>
<td><p>%</p></td>
</tr>
<tr class="row-even"><td><p>Minimum Relative Humidity</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">rmin</span></code></p></td>
<td><p>%</p></td>
</tr>
<tr class="row-odd"><td><p>Specific Humidity</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">sph</span></code></p></td>
<td><p>kg/kg</p></td>
</tr>
<tr class="row-even"><td><p>Surface Radiation</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">srad</span></code></p></td>
<td><p>W/m2</p></td>
</tr>
<tr class="row-odd"><td><p>Wind Direction</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">th</span></code></p></td>
<td><p>Degrees Clockwise from north</p></td>
</tr>
<tr class="row-even"><td><p>Minimum Air Temperature</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">tmmn</span></code></p></td>
<td><p>K</p></td>
</tr>
<tr class="row-odd"><td><p>Maximum Air Temperature</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">tmmx</span></code></p></td>
<td><p>K</p></td>
</tr>
<tr class="row-even"><td><p>Wind Speed</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">vs</span></code></p></td>
<td><p>m/s</p></td>
</tr>
<tr class="row-odd"><td><p>Burning Index</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">bi</span></code></p></td>
<td><p>Dimensionless</p></td>
</tr>
<tr class="row-even"><td><p>Fuel Moisture (100-hr)</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">fm100</span></code></p></td>
<td><p>%</p></td>
</tr>
<tr class="row-odd"><td><p>Fuel Moisture (1000-hr)</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">fm1000</span></code></p></td>
<td><p>%</p></td>
</tr>
<tr class="row-even"><td><p>Energy Release Component</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">erc</span></code></p></td>
<td><p>Dimensionless</p></td>
</tr>
<tr class="row-odd"><td><p>Reference Evapotranspiration (Alfalfa)</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">etr</span></code></p></td>
<td><p>mm</p></td>
</tr>
<tr class="row-even"><td><p>Reference Evapotranspiration (Grass)</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">pet</span></code></p></td>
<td><p>mm</p></td>
</tr>
<tr class="row-odd"><td><p>Vapor Pressure Deficit</p></td>
<td><p><code class="docutils literal notranslate"><span class="pre">vpd</span></code></p></td>
<td><p>kPa</p></td>
</tr>
</tbody>
</table>

In [ ]:
data

In [ ]:
data_ed.loc[station,'co2'].replace(-9999,np.nan).plot()

In [ ]:
imp_mean = IterativeImputer(random_state=0, max_iter=30, missing_values=np.nan,
                            sample_posterior=True,n_nearest_features=100,)
imp_mean.fit(data)
#X = [[np.nan, 2, 3], [4, np.nan, 6], [10, np.nan, 9]]
new_utd = pd.DataFrame(imp_mean.transform(data), columns=data.columns)
data['imp_LE'] = new_utd['LE']

* footprint
* import opentet

* Import remote sensing data to impute
